# 多模态模型微调实战教程

本教程讲解如何对视觉-语言模型进行微调：
- 全参数微调
- 冻结编码器微调
- LoRA 高效微调
- 对比学习微调

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Dict, Optional

from clip import CLIP, CLIPConfig, create_clip_model, clip_loss
from blip import BLIP, BLIPConfig, create_blip_model

## 1. 冻结编码器微调

只训练投影层，保持编码器冻结。

In [ ]:
class FrozenEncoderFineTuner:
    """冻结编码器的微调器"""
    
    def __init__(self, model: CLIP):
        self.model = model
        self._freeze_encoders()
    
    def _freeze_encoders(self):
        """冻结视觉和文本编码器"""
        for param in self.model.vision_encoder.parameters():
            param.requires_grad = False
        for param in self.model.text_encoder.parameters():
            param.requires_grad = False
        
        # 只训练投影层
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.model.parameters())
        print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    
    def get_trainable_params(self):
        return [p for p in self.model.parameters() if p.requires_grad]

model = create_clip_model("small")
finetuner = FrozenEncoderFineTuner(model)

## 2. LoRA 高效微调

In [ ]:
class LoRALayer(nn.Module):
    """LoRA 低秩适配层"""
    
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        self.lora_A = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        nn.init.kaiming_uniform_(self.lora_A)
        nn.init.zeros_(self.lora_B)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (x @ self.lora_A @ self.lora_B) * self.scaling

class LoRALinear(nn.Module):
    """带 LoRA 的线性层"""
    
    def __init__(self, linear: nn.Linear, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)
        
        # 冻结原始权重
        for param in self.linear.parameters():
            param.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x) + self.lora(x)

def apply_lora_to_model(model: nn.Module, rank: int = 4, target_modules: List[str] = None):
    """将 LoRA 应用到模型的指定模块"""
    target_modules = target_modules or ['q_proj', 'v_proj']
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            if any(t in name for t in target_modules):
                parent_name = '.'.join(name.split('.')[:-1])
                child_name = name.split('.')[-1]
                parent = model.get_submodule(parent_name) if parent_name else model
                setattr(parent, child_name, LoRALinear(module, rank))
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"LoRA trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# 测试 LoRA
model = create_clip_model("small")
apply_lora_to_model(model, rank=8)

## 3. 对比学习微调

In [ ]:
class ContrastiveFineTuner:
    """对比学习微调器"""
    
    def __init__(self, model: CLIP, lr: float = 1e-5, weight_decay: float = 0.01):
        self.model = model
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
    
    def train_step(self, images: torch.Tensor, input_ids: torch.Tensor) -> float:
        self.model.train()
        self.optimizer.zero_grad()
        
        image_feat, text_feat, logit_scale = self.model(images, input_ids)
        loss = clip_loss(image_feat, text_feat, logit_scale)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        
        return loss.item()

# 测试训练步骤
model = create_clip_model("small")
trainer = ContrastiveFineTuner(model)

images = torch.randn(4, 3, 224, 224)
texts = torch.randint(0, 49408, (4, 77))
loss = trainer.train_step(images, texts)
print(f"Training loss: {loss:.4f}")

## 4. 学习率调度

In [ ]:
def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps: int, num_training_steps: int):
    """余弦退火学习率调度器"""
    import math
    
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# 示例
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=1000)
print(f"Initial LR: {scheduler.get_last_lr()[0]:.6f}")

## 总结

本教程介绍了多模态模型微调的核心技术：

1. **冻结编码器**: 只训练投影层，减少计算量
2. **LoRA 微调**: 低秩适配，参数高效
3. **对比学习**: 使用 InfoNCE 损失微调
4. **学习率调度**: Warmup + 余弦退火